## DeepSynergy

Author: Kristina Preuer

This Keras script shows how DeepSynergy was evaluated in one cross validation run (executed 5 times - looping over test folds). In this examples fold 0 is used for testing. The script uses 60% of the data  for training (folds 2, 3, 4) and 20% for validation (fold 1). The parameters are loaded with a separate text file (hyperparameters). Validation loss was used to determine the early stopping parameter. After hyperparameter selection the training and validation data was combined (80% = folds 1, 2, 3, 4) and the remaining 20% (fold 0) of the data were used for testing.

The original work was done accordingly with binet (https://github.com/bioinf-jku/binet/tree/master/binet). 

In [1]:
import os, sys

import pandas as pd
import numpy as np
import pickle
import gzip
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt

os.environ["CUDA_VISIBLE_DEVICES"]="3" #specify GPU 
from tensorflow import keras as K
import tensorflow as tf
from tensorflow.keras import backend
from tensorflow.compat.v1.keras.backend import set_session
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

2022-03-31 18:47:11.644656: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0


#### Define parameters for this cross-validation run

In [2]:
hyperparameter_file = 'hyperparameters' # textfile which contains the hyperparameters of the model
data_file = '/home/nidhi/Documents/freelancing/DeepSynergy/data/data_test_fold0_tanh.p.gz' # pickle file which contains the data (produced with normalize.ipynb)

#### Define smoothing functions for early stopping parameter

In [3]:
def moving_average(a, n=3):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

#### Load parameters defining the model

In [4]:
exec(open(hyperparameter_file).read()) 

#### Load data 
tr = 60% of data for training during hyperparameter selection <br>
val = 20% of data for validation during hyperparameter selection

train = tr + val = 80% of data for training during final testing <br>
test = remaining left out 20% of data for unbiased testing 

splitting and normalization was done with normalize.ipynb

In [5]:
file = gzip.open(data_file, 'rb')
X_tr, X_val, X_train, X_test, y_tr, y_val, y_train, y_test = pickle.load(file)
file.close()

#### run set

In [6]:
config = tf.compat.v1.ConfigProto(
         allow_soft_placement=True,
         #gpu_options = tf.GPUOptions(allow_growth=True)
        # gpu_options = tf.compat.v1.GPUOptions(allow_growth = True)
        )

config.gpu_options.allow_growth = True

In [7]:

#set_session(tf.compat.v1.Session()(config=config))

from tensorflow.compat.v1 import InteractiveSession
session = InteractiveSession(config=config)



2022-03-31 18:47:53.973361: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-03-31 18:47:54.337788: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2022-03-31 18:47:54.362088: E tensorflow/stream_executor/cuda/cuda_driver.cc:328] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2022-03-31 18:47:54.362133: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: Olympias
2022-03-31 18:47:54.362146: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: Olympias
2022-03-31 18:47:54.362543: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported 

In [8]:
model = Sequential()
metrics=[tf.keras.metrics.MeanSquaredError(), 
         tf.keras.metrics.RootMeanSquaredError(),
         tf.keras.metrics.MeanAbsoluteError()]
for i in range(len(layers)):
    if i==0:
        model.add(Dense(layers[i], input_shape=(X_tr.shape[1],), activation=act_func, 
                        kernel_initializer='he_normal'))
        model.add(Dropout(float(input_dropout)))
    elif i==len(layers)-1:
        model.add(Dense(layers[i], activation='linear', kernel_initializer="he_normal"))
    else:
        model.add(Dense(layers[i], activation=act_func, kernel_initializer="he_normal"))
        model.add(Dropout(float(dropout)))
    model.compile(loss='mean_squared_error', metrics= metrics, optimizer=K.optimizers.SGD(lr=float(eta), momentum=0.5))

/home/nidhi/miniconda3/lib/python3.8/site-packages/tensorflow/python/keras/optimizer_v2/optimizer_v2.py:374: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  warnings.warn(


Train model

In [9]:
print(X_tr.shape)
print(y_tr.shape)

(27768, 8846)
(27768,)


In [10]:
hist = model.fit(X_tr, y_tr, epochs=15, shuffle=True, batch_size=64, validation_data=(X_val, y_val))
val_loss = hist.history['val_loss']
#model.reset_states()

2022-03-31 18:47:56.989781: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2022-03-31 18:47:57.036634: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 4200000000 Hz


Epoch 1/15
434/434 [==============================] - 115s 260ms/step - loss: 539.1463 - mean_squared_error: 539.1463 - root_mean_squared_error: 23.2195 - mean_absolute_error: 14.7050 - val_loss: 417.3738 - val_mean_squared_error: 417.3738 - val_root_mean_squared_error: 20.4297 - val_mean_absolute_error: 13.8006
Epoch 2/15
434/434 [==============================] - 114s 262ms/step - loss: 480.7771 - mean_squared_error: 480.7771 - root_mean_squared_error: 21.9266 - mean_absolute_error: 13.6957 - val_loss: 382.5614 - val_mean_squared_error: 382.5614 - val_root_mean_squared_error: 19.5592 - val_mean_absolute_error: 13.1412
Epoch 3/15
434/434 [==============================] - 113s 261ms/step - loss: 445.5926 - mean_squared_error: 445.5926 - root_mean_squared_error: 21.1091 - mean_absolute_error: 13.1667 - val_loss: 368.1418 - val_mean_squared_error: 368.1418 - val_root_mean_squared_error: 19.1870 - val_mean_absolute_error: 12.9385
Epoch 4/15
434/434 [==============================] - 113s

In [11]:
model.save("model_15_epochs.h5")
model_15epochs = load_model("model_15_epochs.h5")
predictions_15epochs = model_15epochs.predict(X_test)


In [12]:
predictions_15epochs

array([[ 5.9254904 ],
       [ 2.5368094 ],
       [ 0.16147615],
       ...,
       [ 8.795495  ],
       [13.742191  ],
       [ 4.7897468 ]], dtype=float32)

In [13]:
hist.history

{'loss': [539.1463012695312,
  480.7771301269531,
  445.5926208496094,
  429.6029968261719,
  419.8587646484375,
  410.4975280761719,
  402.4478759765625,
  394.3927307128906,
  385.0713806152344,
  376.6007385253906,
  367.0849609375,
  357.0908203125,
  344.7840576171875,
  334.7314147949219,
  323.0951843261719],
 'mean_squared_error': [539.1463012695312,
  480.7771301269531,
  445.5926208496094,
  429.6029968261719,
  419.8587646484375,
  410.4975280761719,
  402.4478759765625,
  394.3927307128906,
  385.0713806152344,
  376.6007385253906,
  367.0849609375,
  357.0908203125,
  344.7840576171875,
  334.7314147949219,
  323.0951843261719],
 'root_mean_squared_error': [23.219524383544922,
  21.9266300201416,
  21.10906410217285,
  20.726865768432617,
  20.490455627441406,
  20.260738372802734,
  20.06110382080078,
  19.859323501586914,
  19.62323570251465,
  19.406204223632812,
  19.159461975097656,
  18.896846771240234,
  18.568361282348633,
  18.295665740966797,
  17.9748477935791],

Evaluate model on test data

In [14]:
print("Regression metrics from the model")
results = model.evaluate(X_test, y_test, batch_size=128)
print(results)
print("Mean squared Error=", results[1])
print("Root Mean squared Error=", results[2])
print("Mean Absolute Error=", results[3])


Regression metrics from the model
72/72 [==============================] - 7s 91ms/step - loss: 295.7663 - mean_squared_error: 295.7663 - root_mean_squared_error: 17.1979 - mean_absolute_error: 11.5676
[295.7662658691406, 295.7662658691406, 17.197856903076172, 11.567566871643066]
Mean squared Error= 295.7662658691406
Root Mean squared Error= 17.197856903076172
Mean Absolute Error= 11.567566871643066


Synergy Score Prediction

In [15]:
print("Synergy Score Prediction")
predictions = model.predict(X_test)
print("predictions shape:", predictions.shape)

print(predictions)

Synergy Score Prediction
predictions shape: (9108, 1)
[[ 5.9254904 ]
 [ 2.5368094 ]
 [ 0.16147615]
 ...
 [ 8.795495  ]
 [13.742191  ]
 [ 4.7897468 ]]


Classification metrics for model

In [16]:
#generate classification labels.
#Classification threshold = 30 as used in paper.
classification_df = pd.DataFrame(y_test, columns= ["y_test"])

classification_df.loc[classification_df.y_test > 30, "y_label"] = 1
classification_df.loc[classification_df.y_test < 30, "y_label"] = 0

#add predictions to classification_df
classification_df["predictions"] = predictions

classification_df.loc[classification_df.predictions > 30, "predicted_label"] = 1
classification_df.loc[classification_df.predictions < 30, "predicted_label"] = 0

classification_df.head()


,y_test,y_label,predictions,predicted_label
0,4.331695,0.0,5.925490,0.0
1,-8.161129,0.0,2.536809,0.0
2,-5.415934,0.0,0.161476,0.0
3,-7.747474,0.0,0.428431,0.0
4,-14.347426,0.0,-0.895611,0.0


In [17]:
correct_preds = classification_df.index[classification_df.y_label == classification_df.predicted_label].tolist()
accuracy = 100 * len(correct_preds)/classification_df.shape[0]
balanced_accuracy = 100 * balanced_accuracy_score(np.array(classification_df.y_label), np.array(classification_df.predicted_label))
print("accuracy = {}%".format(accuracy))
print("balanced accuracy = {0}%".format(balanced_accuracy))
print("Confusion Matrix:")
print(confusion_matrix(np.array(classification_df.y_label), np.array(classification_df.predicted_label)))



accuracy = 94.96047430830039%
balanced accuracy = 64.84110295314473%
Confusion Matrix:
[[8486   92]
 [ 367  163]]


Accuracy per epoch

In [18]:
model = Sequential()
metrics=[tf.keras.metrics.MeanSquaredError(), 
        tf.keras.metrics.RootMeanSquaredError(),
        tf.keras.metrics.MeanAbsoluteError()]
for i in range(len(layers)):
    if i==0:
        model.add(Dense(layers[i], input_shape=(X_tr.shape[1],), activation=act_func, 
        kernel_initializer='he_normal'))
        
        model.add(Dropout(float(input_dropout)))
        
    elif i==len(layers)-1:
        model.add(Dense(layers[i], activation='linear', kernel_initializer="he_normal"))
    
    else:
        model.add(Dense(layers[i], activation=act_func, kernel_initializer="he_normal"))
        model.add(Dropout(float(dropout)))
        
    model.compile(loss='mean_squared_error', metrics= metrics, optimizer=K.optimizers.SGD(lr=float(eta), momentum=0.5))
model.save("model.h5") 

index = 0
columns = ["MSE", "RMSE", "MAE", "acc", "balanced_acc"]
metrics_df = pd.DataFrame(columns = columns)

for epoch in range(1,50):
    
    model = load_model('model.h5')
    hist = model.fit(X_train, y_train, epochs=1, shuffle=True, batch_size=64, validation_data=(X_test, y_test))

    print("Regression metrics from the model")
    results = model.evaluate(X_test, y_test, batch_size=128)
    predictions2 = model.predict(X_test)
    classification_df = pd.DataFrame(y_test, columns= ["y_test"])

    classification_df.loc[classification_df.y_test > 30, "y_label"] = 1
    classification_df.loc[classification_df.y_test < 30, "y_label"] = 0

    #add predictions to classification_df
    classification_df["predictions"] = predictions2
    classification_df.loc[classification_df.predictions > 30, "predicted_label"] = 1
    classification_df.loc[classification_df.predictions < 30, "predicted_label"] = 0

    correct_preds = classification_df.index[classification_df.y_label == classification_df.predicted_label].tolist()
    accuracy = 100 * len(correct_preds)/classification_df.shape[0]
    balanced_accuracy = 100 * balanced_accuracy_score(np.array(classification_df.y_label), np.array(classification_df.predicted_label))

    #metrics per epoch
    metrics_df.at[index, "MSE"] = results[1]
    metrics_df.at[index, "RMSE"] = results[2]
    metrics_df.at[index, "MAE"] = results[3]
    metrics_df.at[index, "acc"] = accuracy
    metrics_df.at[index, "balanced_acc"] = balanced_accuracy

    index +=1
    print(metrics_df)
    model.save("model.h5")
    print("Saved model to disk")




        

/home/nidhi/miniconda3/lib/python3.8/site-packages/tensorflow/python/keras/optimizer_v2/optimizer_v2.py:374: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  warnings.warn(
2022-03-31 19:15:54.006531: W tensorflow/core/framework/cpu_allocator_impl.cc:80] Allocation of 1309066464 exceeds 10% of free system memory.


579/579 [==============================] - 148s 256ms/step - loss: 507.2066 - mean_squared_error: 507.2066 - root_mean_squared_error: 22.5212 - mean_absolute_error: 14.4532 - val_loss: 395.0641 - val_mean_squared_error: 395.0641 - val_root_mean_squared_error: 19.8762 - val_mean_absolute_error: 13.2381
Regression metrics from the model
72/72 [==============================] - 6s 86ms/step - loss: 395.0641 - mean_squared_error: 395.0641 - root_mean_squared_error: 19.8762 - mean_absolute_error: 13.2381
       MSE     RMSE      MAE      acc balanced_acc
0  395.064  19.8762  13.2381  94.1809           50
Saved model to disk


2022-03-31 19:18:46.836699: W tensorflow/core/framework/cpu_allocator_impl.cc:80] Allocation of 1309066464 exceeds 10% of free system memory.


579/579 [==============================] - 144s 249ms/step - loss: 445.0585 - mean_squared_error: 445.0585 - root_mean_squared_error: 21.0964 - mean_absolute_error: 13.3606 - val_loss: 362.8291 - val_mean_squared_error: 362.8291 - val_root_mean_squared_error: 19.0481 - val_mean_absolute_error: 12.5885
Regression metrics from the model
72/72 [==============================] - 6s 86ms/step - loss: 362.8293 - mean_squared_error: 362.8293 - root_mean_squared_error: 19.0481 - mean_absolute_error: 12.5885
       MSE     RMSE      MAE      acc balanced_acc
0  395.064  19.8762  13.2381  94.1809           50
1  362.829  19.0481  12.5885  94.3127      51.8402
Saved model to disk


2022-03-31 19:21:36.165748: W tensorflow/core/framework/cpu_allocator_impl.cc:80] Allocation of 1309066464 exceeds 10% of free system memory.


579/579 [==============================] - 146s 252ms/step - loss: 417.7011 - mean_squared_error: 417.7011 - root_mean_squared_error: 20.4377 - mean_absolute_error: 12.9394 - val_loss: 349.8542 - val_mean_squared_error: 349.8542 - val_root_mean_squared_error: 18.7044 - val_mean_absolute_error: 12.3419
Regression metrics from the model
72/72 [==============================] - 7s 92ms/step - loss: 349.8542 - mean_squared_error: 349.8542 - root_mean_squared_error: 18.7044 - mean_absolute_error: 12.3419
       MSE     RMSE      MAE      acc balanced_acc
0  395.064  19.8762  13.2381  94.1809           50
1  362.829  19.0481  12.5885  94.3127      51.8402
2  349.854  18.7044  12.3419  94.4554      57.3151
Saved model to disk


2022-03-31 19:24:27.540121: W tensorflow/core/framework/cpu_allocator_impl.cc:80] Allocation of 1309066464 exceeds 10% of free system memory.


579/579 [==============================] - 149s 257ms/step - loss: 404.3275 - mean_squared_error: 404.3275 - root_mean_squared_error: 20.1079 - mean_absolute_error: 12.7434 - val_loss: 339.8323 - val_mean_squared_error: 339.8323 - val_root_mean_squared_error: 18.4345 - val_mean_absolute_error: 12.0948
Regression metrics from the model
72/72 [==============================] - 6s 89ms/step - loss: 339.8324 - mean_squared_error: 339.8324 - root_mean_squared_error: 18.4345 - mean_absolute_error: 12.0948
       MSE     RMSE      MAE      acc balanced_acc
0  395.064  19.8762  13.2381  94.1809           50
1  362.829  19.0481  12.5885  94.3127      51.8402
2  349.854  18.7044  12.3419  94.4554      57.3151
3  339.832  18.4345  12.0948  94.5323      57.2674
Saved model to disk


2022-03-31 19:27:22.556679: W tensorflow/core/framework/cpu_allocator_impl.cc:80] Allocation of 1309066464 exceeds 10% of free system memory.


530/579 [==========================>...] - ETA: 12s - loss: 393.3036 - mean_squared_error: 393.3036 - root_mean_squared_error: 19.8319 - mean_absolute_error: 12.5260

KeyboardInterrupt: 

Saving results to disk for plotting.

In [ ]:
# #save predictions to disk for plotting results.
# predictions_df = pd.DataFrame(predictions)
# predictions_df.to_csv("/home/nidhi/Documents/freelancing/DeepSynergy/data/Results/predictions.csv")

In [ ]:
predictions2_df

In [ ]:
#save results of model training version 2 (WITH ACCURACY METRICS PER EPOCH)
predictions2_df = pd.DataFrame(predictions2)
predictions2_df.to_csv("/home/nidhi/Documents/freelancing/DeepSynergy/data/Results/predictions_per_epoch_acc_50_epochs.csv")

In [ ]:
#save metrics_df 
metrics_df.to_csv("/home/nidhi/Documents/freelancing/DeepSynergy/data/Results/metrics_per_epoch_50epochs.csv")

#### smooth validation loss for early stopping parameter determination

In [ ]:
average_over = 2 #was 15 earlier. We only ran 2 epochs. So, threw error.
mov_av = moving_average(np.array(val_loss), average_over)
print(mov_av)
smooth_val_loss = np.pad(mov_av, int(average_over/2), mode='edge')
epo = np.argmin(smooth_val_loss)


#### determine model performance for methods comparison 

In [ ]:
hist = model.fit(X_train, y_train, epochs=2, shuffle=True, batch_size=64, validation_data=(X_test, y_test))
test_loss = hist.history['val_loss']

#### plot performance 

In [ ]:
fig, ax = plt.subplots(figsize=(16,8))
ax.plot(val_loss, label='validation loss')
ax.plot(smooth_val_loss, label='smooth validation loss')
ax.plot(test_loss, label='test loss')
ax.legend()
plt.show()